# Repository note
This is a code-only copy of an exploratory notebook. Saved outputs were removed for public sharing. The required CSV datasets are not included; update the local/Colab paths in the cells before running. The original execution environment and full dataset provenance were not supplied, so a fresh run is not guaranteed to reproduce prior results.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Technique Generalization Testing
## Leave-One-Technique-Out (LOTO) Evaluation
### Transformer-Based Network Steganography Detector

---

**Purpose:** Evaluate whether the model generalizes to steganographic techniques it has never seen during training.

**Methodology:** For each of the 3 splits, a completely fresh model is trained on 2 steganographic techniques plus normal traffic, then evaluated on the held-out 3rd technique.

| Dataset Label | Binary Label | Description |
|--------------|-------------|-------------|
| Technique 0 | 0 | Normal (benign) traffic |
| Technique 1 | 1 | Header Manipulation |
| Technique 2 | 1 | Timing Obfuscation |
| Technique 3 | 1 | Flow Blending |

| Split | Train On | Test On (Unseen) |
|-------|----------|------------------|
| 1 | Header Manipulation + Flow Blending + Normal | Timing Obfuscation + Normal |
| 2 | Header Manipulation + Timing Obfuscation + Normal | Flow Blending + Normal |
| 3 | Timing Obfuscation + Flow Blending + Normal | Header Manipulation + Normal |

## Section 1 — Imports
Environment Setup

In [ ]:
import torch
import pandas as pd
import torch.nn as nn
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

print("All libraries imported successfully")

## Section 2 — Device Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## Section 3 — Load Dataset & Verify Structure

In [ ]:

# TECHNIQUES_MAPPING  = {0: 'Normal Traffic', 1: 'Header Manipulation', 2: 'Timing Obfuscation', 3: 'Flow Blending'}
# PROTOCOLS_MAPPING = {0: 'DNS', 1: 'ICMP', 2: 'TCP', 3: 'UDP'}


CSV_PATH = r"/content/drive/MyDrive/Folder-FYP/Network_steganography_Project/Dataset/preprocessed_master_dataset.csv"

df = pd.read_csv(CSV_PATH)

print(" Dataset loaded")

print(f"   Techniques    : {sorted(df['label_technique'].unique())}")
print(f"   Binary labels : {sorted(df['label_binary'].unique())}")
print()

print("📊 Samples per technique:")
print(df['label_technique'].value_counts().sort_index())
print()

print("📊 Binary class balance PER TECHNIQUE:")
pivot = df.groupby('label_technique')['label_binary'].value_counts().unstack(fill_value=0)
pivot.columns = [f'label={c}' for c in pivot.columns]
print(pivot)
print()

# ── Key structural check ──────────────────────────────────────
normal_techs = sorted(df[df['label_binary'] == 0]['label_technique'].unique())
stego_techs  = sorted(df[df['label_binary'] == 1]['label_technique'].unique())
print(f"   Normal traffic technique labels : {normal_techs}")
print(f"   Steganography technique labels  : {stego_techs}")
print()
print(" LOTO will use: normal=technique 0 (always included), stego=techniques 1/2/3")



## Section 4 — Model Architecture

The `FeatureTokenTransformer` treats each input feature as an individual token, applies multi-head self-attention across all features, and produces three simultaneous outputs: binary detection, technique classification, and protocol classification.

Architecture  : 4 × AttentionBlock (embed_dim=128, heads=8) + 3 output heads'

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, embed_dim=128, heads=8):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, heads, batch_first=True)
        self.norm1     = nn.LayerNorm(embed_dim)
        self.ff        = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Linear(256, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, attn_weights = self.attention(x, x, x)
        x = self.norm1(x + attn_output)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x, attn_weights


class FeatureTokenTransformer(nn.Module):
    def __init__(self, num_features=225, embed_dim=128):
        super().__init__()
        self.embedding   = nn.Linear(1, embed_dim)
        self.blocks      = nn.ModuleList([AttentionBlock(embed_dim) for _ in range(4)])
        self.shared      = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.binary_head = nn.Linear(128, 1)
        self.tech_head   = nn.Linear(128, 4)
        self.proto_head  = nn.Linear(128, 4)

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.embedding(x)
        attention_maps = []
        for block in self.blocks:
            x, attn = block(x)
            attention_maps.append(attn)
        x      = torch.mean(x, dim=1)
        shared = self.shared(x)
        binary = self.binary_head(shared)
        tech   = self.tech_head(shared)
        proto  = self.proto_head(shared)
        return binary, tech, proto, attention_maps


print("✅ Model architecture defined")
_tmp = FeatureTokenTransformer()
print(f"   Total parameters: {sum(p.numel() for p in _tmp.parameters()):,}")
del _tmp

---
## Section 5— Training Configuration & Helper Functions

###  Normal traffic (technique 0) is included in EVERY train and test split
###  Confusion matrix uses only labels actually present in the data

In [ ]:
# ─────────────────────────────────────────────
# TRAINING CONFIGURATION
# ─────────────────────────────────────────────
EPOCHS     = 10      # increase to 20-30 for stronger results
BATCH_SIZE = 64
LR         = 1e-4
NORMAL_TECHNIQUE = 0   # technique label for benign/normal traffic
# ─────────────────────────────────────────────

tech_names = {
    0: "Normal Traffic",
    1: "Header Manipulation",
    2: "Timing Obfuscation",
    3: "Flow Blending"
}


def make_loaders(train_df, test_df, batch_size=BATCH_SIZE):
    """Build PyTorch DataLoaders from train/test DataFrames."""
    drop_cols = ["label_binary", "label_technique", "label_protocol_enc"]

    X_tr = torch.tensor(train_df.drop(drop_cols, axis=1).values, dtype=torch.float32)
    X_te = torch.tensor(test_df.drop(drop_cols, axis=1).values,  dtype=torch.float32)

    def _labels(d):
        yb = torch.tensor(d["label_binary"].values,       dtype=torch.float32).unsqueeze(1)
        yt = torch.tensor(d["label_technique"].values,    dtype=torch.long)
        yp = torch.tensor(d["label_protocol_enc"].values, dtype=torch.long)
        return yb, yt, yp

    tr_loader = DataLoader(TensorDataset(X_tr, *_labels(train_df)),
                           batch_size=batch_size, shuffle=True)
    te_loader = DataLoader(TensorDataset(X_te, *_labels(test_df)),
                           batch_size=batch_size)
    return tr_loader, te_loader


def train_model(train_loader, epochs=EPOCHS, lr=LR, verbose=True):
    """Initialise a fresh model and train it. Returns trained model."""
    model = FeatureTokenTransformer().to(device)
    criterion_binary = nn.BCEWithLogitsLoss()
    criterion_multi  = nn.CrossEntropyLoss()
    optimizer        = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for x, yb, yt, yp in train_loader:
            x, yb, yt, yp = x.to(device), yb.to(device), yt.to(device), yp.to(device)
            optimizer.zero_grad()
            pb, pt, pp, _ = model(x)
            loss = (criterion_binary(pb, yb) +
                    criterion_multi(pt, yt) +
                    criterion_multi(pp, yp))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        if verbose and (epoch % 2 == 0 or epoch == 1 or epoch == epochs):
            print(f"   Epoch {epoch:>3}/{epochs}  |  Loss: {epoch_loss/len(train_loader):.4f}")
    return model


def evaluate_model(model, test_loader):
    """Evaluate model. Returns dict with predictions and accuracies."""
    model.eval()
    preds_b, truth_b = [], []
    preds_p, truth_p = [], []

    with torch.no_grad():
        for x, yb, yt, yp in test_loader:
            x = x.to(device)
            pb, pt, pp, _ = model(x)

            pb_pred = (torch.sigmoid(pb) > 0.5).cpu().numpy().flatten()
            preds_b.extend(pb_pred.tolist())
            truth_b.extend(yb.numpy().flatten().tolist())

            pp_pred = torch.argmax(pp, dim=1).cpu().numpy()
            preds_p.extend(pp_pred.tolist())
            truth_p.extend(yp.numpy().tolist())

    return {
        "binary_acc"  : accuracy_score(truth_b, preds_b),
        "protocol_acc": accuracy_score(truth_p, preds_p),
        "truth_b"     : [int(x) for x in truth_b],
        "preds_b"     : [int(x) for x in preds_b],
        "truth_p"     : [int(x) for x in truth_p],
        "preds_p"     : [int(x) for x in preds_p],
    }


def print_results(results, train_techs, test_tech):
    """Pretty-print results — robust to any class distribution."""
    train_str = " + ".join(f"{tech_names[t]} ({t})" for t in train_techs)
    test_str  = f"{tech_names[test_tech]} ({test_tech})"

    print(f"\n{'='*62}")
    print(f" Train : {train_str} + Normal (0)")
    print(f" Test  : {test_str} + Normal (0)  ← UNSEEN stego during training")
    print(f"{'='*62}")
    print(f"  🎯 Binary Detection Accuracy : {results['binary_acc']*100:.2f}%")
    print(f"  🔗 Protocol Accuracy         : {results['protocol_acc']*100:.2f}%")

    # Binary report — only use classes that are actually present
    present_b = sorted(set(results['truth_b']))
    b_names   = {0: "Normal", 1: "Steganography"}
    b_labels  = [b_names[c] for c in present_b]

    print(f"\n  📊 Binary Classification Report (Detection):")
    print(classification_report(
        results['truth_b'], results['preds_b'],
        labels=present_b,
        target_names=b_labels,
        digits=4,
        zero_division=0
    ))

    print(f"  📊 Protocol Classification Report:")
    print(classification_report(
        results['truth_p'], results['preds_p'],
        digits=4,
        zero_division=0
    ))


print("✅ All helper functions defined")
print(f"   EPOCHS={EPOCHS} | BATCH_SIZE={BATCH_SIZE} | LR={LR}")

In [ ]:
import numpy as np

# Define LOTO_SPLITS (moved from a later cell to resolve NameError)
LOTO_SPLITS = [
    ([1, 3], 2),   # Train: Tech1 + Tech3 + Normal  → Test: Tech2 + Normal
    ([1, 2], 3),   # Train: Tech1 + Tech2 + Normal  → Test: Tech3 + Normal
    ([2, 3], 1),   # Train: Tech2 + Tech3 + Normal  → Test: Tech1 + Normal
]

# Define normal_df (moved from a later cell to resolve NameError for normal_train/normal_test)
# Assuming df and NORMAL_TECHNIQUE are defined in preceding cells
normal_df = df[df["label_technique"] == NORMAL_TECHNIQUE].copy()

seeds = [0, 1, 2, 3, 4]
seed_results = {1: [], 2: [], 3: []}  # split_idx -> list of binary_acc across seeds

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    for split_idx, (train_techs, test_tech) in enumerate(LOTO_SPLITS, 1):
        stego_train = df[df["label_technique"].isin(train_techs)].copy()
        # Corrected: use normal_df instead of undefined normal_train
        train_df = pd.concat([stego_train, normal_df], ignore_index=True)
        stego_test = df[df["label_technique"] == test_tech].copy()
        # Corrected: use normal_df instead of undefined normal_test
        test_df = pd.concat([stego_test, normal_df], ignore_index=True)
        train_loader, test_loader = make_loaders(train_df, test_df)
        model = train_model(train_loader, epochs=EPOCHS, verbose=False)
        results = evaluate_model(model, test_loader)
        seed_results[split_idx].append(results["binary_acc"] * 100)
        print(f"Seed {seed}, Split {split_idx}: {results['binary_acc']*100:.2f}%")

print("\n=== Per-split mean ± std across 5 seeds ===")
for split_idx in [1, 2, 3]:
    vals = seed_results[split_idx]
    print(f"Split {split_idx}: {np.mean(vals):.2f}% ± {np.std(vals):.2f}%")

all_vals = [v for split_vals in seed_results.values() for v in split_vals]
print(f"\nOverall mean ± std (15 runs): {np.mean(all_vals):.2f}% ± {np.std(all_vals):.2f}%")

In [ ]:
def get_attention_by_category(model, test_loader, feature_names, categories):
    """categories: dict mapping feature_name -> 'header'/'timing'/'flow'"""
    model.eval()
    block_cat_weights = {b: {'header': [], 'timing': [], 'flow': []} for b in range(4)}
    with torch.no_grad():
        for x, yb, yt, yp in test_loader:
            x = x.to(device)
            _, _, _, attn_maps = model(x)
            for b, attn in enumerate(attn_maps):
                # attn shape: (batch, seq_len, seq_len) — average over query dim
                avg_attn = attn.mean(dim=1).cpu().numpy()  # (batch, seq_len)
                for cat in ['header', 'timing', 'flow']:
                    idxs = [i for i, f in enumerate(feature_names) if categories.get(f) == cat]
                    if idxs:
                        block_cat_weights[b][cat].append(avg_attn[:, idxs].mean())
    for b in range(4):
        for cat in ['header', 'timing', 'flow']:
            vals = block_cat_weights[b][cat]
            print(f"Block {b+1}, {cat}: {np.mean(vals):.3f}")
    return block_cat_weights

In [ ]:
!pip install captum
from captum.attr import IntegratedGradients

def get_integrated_gradients(model, test_loader, feature_names, categories):
    ig = IntegratedGradients(lambda x: model(x)[0])  # binary output only
    cat_attr = {'header': [], 'timing': [], 'flow': []}
    for x, yb, yt, yp in test_loader:
        x = x.to(device).requires_grad_()
        attributions = ig.attribute(x, target=None, n_steps=20)
        attr_np = attributions.abs().mean(dim=0).cpu().detach().numpy()
        for cat in ['header', 'timing', 'flow']:
            idxs = [i for i, f in enumerate(feature_names) if categories.get(f) == cat]
            if idxs:
                cat_attr[cat].append(attr_np[idxs].mean())
        break  # one batch is enough for a sample check
    for cat in ['header', 'timing', 'flow']:
        print(f"{cat}: {np.mean(cat_attr[cat]):.4f}")

In [ ]:
feature_names = [c for c in df.columns if c not in ["label_technique", "label_binary", "label_protocol_enc"]]

def categorize(f):
    timing_kw = ['iat_', 'ipt_', 'jitter', 'periodicity', 'timing_regularity', 'duration', 'autocorr']
    header_kw = ['syn_count', 'ack_count', 'psh_count', 'fin_count', 'rst_count', 'win_size', 'seq_',
                 'checksum', 'icmp_type', 'icmp_code', 'icmp_id', 'icmp_seq', 'ip_ttl', 'ttl_values',
                 'ip_id_reuse_rate', 'id_entropy', 'src_port', 'dst_port', 'ans_resource_record',
                 'dns_top_level_domain', 'dns_second_level_domain', 'fwd_header_len', 'icmp_covert_id_count']
    flow_kw = ['pkt_len', 'pkt_size', 'flow_byt', 'flow_pkts', 'payload_len', 'payload_byte',
               'payload_compress', 'payload_printable', 'payload_null', 'payload_bigram',
               'flow_total_bytes', 'total_bytes', 'receiving_bytes', 'sending_bytes', 'packets_len',
               'subflow_fwd', 'totlen_fwd_pkts', 'tot_fwd_pkts', 'dns_domain_name', 'gram_domain',
               'character_entropy', 'character_distribution', 'numerical_percentage',
               'vowels_consonant_ratio', 'conv_freq_vowels_consonants', 'max_continuous',
               'distinct_A_records', 'pkt_rate', 'flow_kl_divergence']
    for kw in timing_kw:
        if kw in f: return 'timing'
    for kw in header_kw:
        if kw in f: return 'header'
    for kw in flow_kw:
        if kw in f: return 'flow'
    return 'unmatched'

categories = {f: categorize(f) for f in feature_names}
from collections import Counter
print(Counter(categories.values()))
print("\nUnmatched features:", [f for f, c in categories.items() if c == 'unmatched'])

In [ ]:
manual_overrides = {
    'ack_unique_count': 'header', 'icmp_invalid_type_ratio': 'header',
    'icmp_non_echo_ratio': 'header', 'icmp_nonstandard_type_count': 'header',
    'icmp_nonzero_code_ratio': 'header', 'dns_subdomain_name_length': 'header',
    'packets_rate': 'timing',
    'retransmission_count': 'flow', 'segment_not_captured_count': 'flow',
    'icmp_nonstd_payload_ratio': 'flow', 'icmp_orphan_request_ratio': 'flow',
    'pkt_uniform_size_ratio': 'flow', 'fwd_pkts_s': 'flow',
}
categories.update(manual_overrides)
print(Counter(categories.values()))

In [ ]:
# pick any test_loader from your LOTO results, e.g. split 1
# Reproduce one LOTO split to get a model, train_df, and test_df if all_results is not yet populated.

# Assuming LOTO_SPLITS, df, normal_df, EPOCHS, NORMAL_TECHNIQUE, train_model, evaluate_model, make_loaders are defined.

# Define variables for a single LOTO split to ensure they are available
# We'll use the first split from LOTO_SPLITS for demonstration purposes
split_idx, (train_techs, test_tech) = 1, LOTO_SPLITS[0]

stego_train = df[df["label_technique"].isin(train_techs)].copy()
train_df = pd.concat([stego_train, normal_df], ignore_index=True)
stego_test = df[df["label_technique"] == test_tech].copy()
test_df = pd.concat([stego_test, normal_df], ignore_index=True)
train_loader_demo, test_loader_demo = make_loaders(train_df, test_df)

# Train a model for this demo split
model_demo = train_model(train_loader_demo, epochs=EPOCHS, verbose=False)

# Use the model_demo instead of all_results[0]['model']
block_weights = get_attention_by_category(model_demo, test_loader_demo, feature_names, categories)

In [ ]:
get_integrated_gradients(all_results[0]['model'], test_loader_demo, feature_names, categories)

In [ ]:
# Train one model fresh (avoids depending on lost all_results)
train_techs, test_tech = LOTO_SPLITS[0]
stego_train = df[df["label_technique"].isin(train_techs)].copy()
train_df = pd.concat([stego_train, normal_df], ignore_index=True)
stego_test = df[df["label_technique"] == test_tech].copy()
test_df = pd.concat([stego_test, normal_df], ignore_index=True)
train_loader, test_loader = make_loaders(train_df, test_df)
demo_model = train_model(train_loader, epochs=EPOCHS, verbose=False)
print("Fresh model trained on Split 1 for interpretability analysis.")

def get_attention_share_by_label(model, test_loader, feature_names, categories):
    model.eval()
    idx_by_cat = {c: [i for i, f in enumerate(feature_names) if categories.get(f) == c]
                  for c in ['header', 'timing', 'flow']}
    block_label_shares = {b: {'normal': [], 'covert': []} for b in range(4)}

    with torch.no_grad():
        for x, yb, yt, yp in test_loader:
            x = x.to(device)
            _, _, _, attn_maps = model(x)
            yb_flat = yb.numpy().flatten()
            for b, attn in enumerate(attn_maps):
                avg_attn = attn.mean(dim=1).cpu().numpy()  # (batch, 225)
                cat_sums = {c: avg_attn[:, idx_by_cat[c]].sum(axis=1) for c in idx_by_cat}
                total = sum(cat_sums.values())
                shares = {c: cat_sums[c] / total for c in cat_sums}
                for i, lbl in enumerate(yb_flat):
                    key = 'covert' if lbl == 1 else 'normal'
                    block_label_shares[b][key].append({c: shares[c][i] for c in shares})

    for b in range(4):
        for key in ['normal', 'covert']:
            entries = block_label_shares[b][key]
            means = {c: np.mean([e[c] for e in entries]) for c in ['header', 'timing', 'flow']}
            print(f"Block {b+1}, {key}: header={means['header']:.3f}, timing={means['timing']:.3f}, flow={means['flow']:.3f}")

get_attention_share_by_label(demo_model, test_loader, feature_names, categories)

---
## Section 6 — LOTO Evaluation Loop

Each split trains a **completely fresh model** — random weights, no shared state — so results genuinely reflect generalization to unseen techniques.

> **Runtime:** Approximately 3–5 minutes per split on GPU.

In [ ]:
# Each tuple: (train_stego_techniques, test_stego_technique)
# Normal traffic (technique 0) is ALWAYS added to both train and test
LOTO_SPLITS = [
    ([1, 3], 2),   # Train: Tech1 + Tech3 + Normal  → Test: Tech2 + Normal
    ([1, 2], 3),   # Train: Tech1 + Tech2 + Normal  → Test: Tech3 + Normal
    ([2, 3], 1),   # Train: Tech2 + Tech3 + Normal  → Test: Tech1 + Normal
]

normal_df = df[df["label_technique"] == NORMAL_TECHNIQUE].copy()
print(f"📋 Normal traffic samples available: {len(normal_df)}")
print()
print("📋 LOTO Splits (normal traffic included in every split):")
for i, (train_techs, test_tech) in enumerate(LOTO_SPLITS, 1):
    train_names = " + ".join(f"{tech_names[t]}({t})" for t in train_techs)
    print(f"  Split {i}: Train [{train_names} + Normal(0)]  →  Test [{tech_names[test_tech]}({test_tech}) + Normal(0)]")

print(f"\n  Training epochs per split : {EPOCHS}")
print(f"  Batch size                : {BATCH_SIZE}")
print(f"  Learning rate             : {LR}")

## Section 7 — Run All 3 LOTO Splits
> ⏳ Trains 3 separate fresh models. Each split takes a few minutes.

In [ ]:
all_results = []

for split_idx, (train_techs, test_tech) in enumerate(LOTO_SPLITS, 1):

    print(f"\n{'#'*62}")
    print(f"  SPLIT {split_idx}/3 — Train on {train_techs} + Normal, Test on {test_tech} + Normal")
    print(f"{'#'*62}")

    # ── 1. Build train set: 2 stego techniques + ALL normal traffic ───
    stego_train = df[df["label_technique"].isin(train_techs)].copy()
    train_df    = pd.concat([stego_train, normal_df], ignore_index=True)

    # ── 2. Build test set: unseen stego technique + ALL normal traffic ─
    stego_test = df[df["label_technique"] == test_tech].copy()
    test_df    = pd.concat([stego_test, normal_df], ignore_index=True)

    print(f"  Train: {len(stego_train)} stego + {len(normal_df)} normal = {len(train_df)} total")
    print(f"  Test : {len(stego_test)} stego  + {len(normal_df)} normal = {len(test_df)} total")
    print(f"  Test binary class balance: {test_df['label_binary'].value_counts().to_dict()}")

    # ── 3. Build DataLoaders ──────────────────────────────────────────
    train_loader, test_loader = make_loaders(train_df, test_df)

    # ── 4. Train a FRESH model ────────────────────────────────────────
    print(f"\n  🏋️  Training fresh model...")
    model = train_model(train_loader, epochs=EPOCHS, verbose=True)

    # ── 5. Evaluate on the unseen technique ──────────────────────────
    print(f"\n  🧪 Evaluating on unseen stego technique {test_tech} + normal...")
    results = evaluate_model(model, test_loader)

    # ── 6. Print detailed results ─────────────────────────────────────
    print_results(results, train_techs, test_tech)

    # ── 7. Store for summary ──────────────────────────────────────────
    all_results.append({
        "split"       : split_idx,
        "train_techs" : train_techs,
        "test_tech"   : test_tech,
        "binary_acc"  : results["binary_acc"],
        "protocol_acc": results["protocol_acc"],
        "results"     : results,
        "model"       : model,
    })

print("\n" + "="*62)
print("  ✅ All 3 LOTO splits completed!")
print("="*62)

## Section 8 — Summary Table

In [ ]:
print("\n LOTO  (Leave-One-Technique-Out) EVALUATION SUMMARY")
print("="*78)
print(f"{'Sp':<4} {'Train On (Stego)':<36} {'Test On (Stego)':<22} {'Binary Acc':>10} {'Proto Acc':>10}")
print("-"*78)

binary_accs, proto_accs = [], []

for r in all_results:
    train_full = " + ".join(tech_names[t] for t in r['train_techs'])
    test_full  = tech_names[r['test_tech']]
    b_acc = r['binary_acc'] * 100
    p_acc = r['protocol_acc'] * 100
    binary_accs.append(b_acc)
    proto_accs.append(p_acc)
    print(f"  {r['split']}   {train_full:<36} {test_full:<22} {b_acc:>9.2f}%  {p_acc:>9.2f}%")

print("-"*78)
print(f"{'MEAN':>62} {np.mean(binary_accs):>9.2f}%  {np.mean(proto_accs):>9.2f}%")
print(f"{'STD':>62} {np.std(binary_accs):>9.2f}%  {np.std(proto_accs):>9.2f}%")
print("="*78)
print()
print(f" Mean Binary Detection Accuracy (LOTO): {np.mean(binary_accs):.2f}% ± {np.std(binary_accs):.2f}%")
print()
print("💡 Interpretation:")
print("   Binary Acc = model's ability to detect stego vs normal on an UNSEEN technique")
print("   High across all 3 splits = true generalization ")

## Section 9 — Confusion Matrices (Binary Detection)
### Uses only labels present in each split

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Binary Detection Confusion Matrices — LOTO", fontsize=14, fontweight='bold')

for i, r in enumerate(all_results):
    truth = r['results']['truth_b']
    preds = r['results']['preds_b']

    # Only use labels present in this split's test set
    present_labels = sorted(set(truth))
    label_names    = ["Normal" if l == 0 else "Stego" for l in present_labels]

    cm   = confusion_matrix(truth, preds, labels=present_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')

    train_str = "+".join(str(t) for t in r['train_techs'])
    axes[i].set_title(
        f"Split {r['split']}: Train Tech {train_str}+Normal\n"
        f"Test: {tech_names[r['test_tech']]} + Normal\n"
        f"Binary Acc: {r['binary_acc']*100:.1f}%",
        fontsize=8
    )

plt.tight_layout()
plt.savefig("loto_confusion_matrices.png", bbox_inches='tight', dpi=150)
plt.show()
print("Confusion matrices saved as loto_confusion_matrices.png")

## Section 10 — Bar Chart: Accuracy per Split

In [ ]:
labels = [
    f"Split {r['split']}\n(Test: {tech_names[r['test_tech']]})"
    for r in all_results
]
b_vals = [r['binary_acc'] * 100 for r in all_results]
p_vals = [r['protocol_acc'] * 100 for r in all_results]

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, b_vals, width, label='Binary Detection Acc', color='steelblue')
bars2 = ax.bar(x + width/2, p_vals, width, label='Protocol Acc',          color='darkorange')

ax.set_ylabel('Accuracy (%)')
ax.set_title('LOTO Evaluation — Accuracy per Split', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 108)
ax.axhline(y=np.mean(b_vals), color='steelblue', linestyle='--', alpha=0.6,
           label=f'Mean Binary Acc ({np.mean(b_vals):.1f}%)')
ax.legend()
ax.yaxis.grid(True, alpha=0.3)

for bar in bars1:
    ax.annotate(f'{bar.get_height():.1f}%',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.1f}%',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("loto_accuracy_chart.png", bbox_inches='tight', dpi=150)
plt.show()
print(" Chart saved as loto_accuracy_chart.png")

# **Single Task Transformer**

In [ ]:
class SingleTaskTransformer(nn.Module):
    def __init__(self, num_features=225, embed_dim=128):
        super().__init__()
        self.embedding = nn.Linear(1, embed_dim)
        self.blocks = nn.ModuleList([AttentionBlock(embed_dim) for _ in range(4)])
        self.shared = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.3)
        )
        self.binary_head = nn.Linear(128, 1)

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.embedding(x)
        for block in self.blocks:
            x, _ = block(x)
        x = torch.mean(x, dim=1)
        shared = self.shared(x)
        return self.binary_head(shared)


def train_single_task(train_loader, epochs=EPOCHS, lr=LR):
    model = SingleTaskTransformer().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(1, epochs + 1):
        for x, yb, yt, yp in train_loader:
            x, yb = x.to(device), yb.to(device)
            optimizer.zero_grad()
            pb = model(x)
            loss = criterion(pb, yb)
            loss.backward()
            optimizer.step()
    return model


def evaluate_single_task(model, test_loader):
    model.eval()
    preds_b, truth_b = [], []
    with torch.no_grad():
        for x, yb, yt, yp in test_loader:
            x = x.to(device)
            pb = model(x)
            preds_b.extend((torch.sigmoid(pb) > 0.5).cpu().numpy().flatten().tolist())
            truth_b.extend(yb.numpy().flatten().tolist())
    return accuracy_score(truth_b, preds_b)


single_task_results = []
for split_idx, (train_techs, test_tech) in enumerate(LOTO_SPLITS, 1):
    stego_train = df[df["label_technique"].isin(train_techs)].copy()
    train_df = pd.concat([stego_train, normal_df], ignore_index=True)
    stego_test = df[df["label_technique"] == test_tech].copy()
    test_df = pd.concat([stego_test, normal_df], ignore_index=True)
    train_loader, test_loader = make_loaders(train_df, test_df)
    st_model = train_single_task(train_loader)
    acc = evaluate_single_task(st_model, test_loader)
    print(f"Split {split_idx} (held-out {tech_names[test_tech]}): Single-task binary acc = {acc*100:.2f}%")
    single_task_results.append(acc)

print(f"\nMean single-task binary accuracy: {np.mean(single_task_results)*100:.2f}%")

In [ ]:
# === Full fix + rerun + check, all in one cell ===

def evaluate_model(model, test_loader):
    model.eval()
    preds_b, truth_b, preds_t, truth_t, preds_p, truth_p = [], [], [], [], [], []
    with torch.no_grad():
        for x, yb, yt, yp in test_loader:
            x = x.to(device)
            pb, pt, pp, _ = model(x)
            preds_b.extend((torch.sigmoid(pb) > 0.5).cpu().numpy().flatten().tolist())
            truth_b.extend(yb.numpy().flatten().tolist())
            preds_t.extend(torch.argmax(pt, dim=1).cpu().numpy().tolist())
            truth_t.extend(yt.numpy().tolist())
            preds_p.extend(torch.argmax(pp, dim=1).cpu().numpy().tolist())
            truth_p.extend(yp.numpy().tolist())
    return {
        "binary_acc": accuracy_score(truth_b, preds_b),
        "protocol_acc": accuracy_score(truth_p, preds_p),
        "technique_acc": accuracy_score(truth_t, preds_t),
        "truth_t": [int(x) for x in truth_t], "preds_t": [int(x) for x in preds_t],
        "truth_b": [int(x) for x in truth_b], "preds_b": [int(x) for x in preds_b],
        "truth_p": [int(x) for x in truth_p], "preds_p": [int(x) for x in preds_p],
    }

all_results = []
for split_idx, (train_techs, test_tech) in enumerate(LOTO_SPLITS, 1):
    print(f"Running split {split_idx}/3...")
    stego_train = df[df["label_technique"].isin(train_techs)].copy()
    train_df = pd.concat([stego_train, normal_df], ignore_index=True)
    stego_test = df[df["label_technique"] == test_tech].copy()
    test_df = pd.concat([stego_test, normal_df], ignore_index=True)
    train_loader, test_loader = make_loaders(train_df, test_df)
    model = train_model(train_loader, epochs=EPOCHS, verbose=False)
    results = evaluate_model(model, test_loader)
    all_results.append({
        "split": split_idx, "train_techs": train_techs, "test_tech": test_tech,
        "binary_acc": results["binary_acc"], "protocol_acc": results["protocol_acc"],
        "results": results, "model": model,
    })
    print(f"  Binary: {results['binary_acc']*100:.2f}% | Protocol: {results['protocol_acc']*100:.2f}% | Technique: {results['technique_acc']*100:.2f}%")

print("\nKeys check:", all_results[0]['results'].keys())

print("\n=== Technique Head Results ===")
for r in all_results:
    print(f"\nSplit {r['split']} — held-out: {tech_names[r['test_tech']]}")
    print(f"Technique Accuracy: {r['results']['technique_acc']*100:.2f}%")
    cm = confusion_matrix(r['results']['truth_t'], r['results']['preds_t'])
    print("Confusion matrix (rows=truth 0-3, cols=pred 0-3):")
    print(cm)

---
## Section 11 — Thesis Results Table

Copy this table directly into your thesis Testing chapter after filling in your run values.

---

**Table X: Leave-One-Technique-Out (LOTO) Evaluation Results**


| Split | Trained On                                               **| Tested On (Unseen)|** Binary Acc | Protocol Acc |


|-------|-----------|-------------------|-----------|--------|----------|------------|--------------|

| 1 | Header Manipulation + Flow Blending + Normal      **|Timing Obfuscation + Normal|** 90.06%| 92.8% |

| 2 | Header Manipulation + Timing Obfuscation + Normal **|Flow Blending + Normal|** 99.0% | 91.1% |

| 3 | Timing Obfuscation + Flow Blending + Normal       **|Header Manipulation + Normal|** 93.6% | 98.9% |


---

**Thesis Paragraph (Testing Chapter):**

> To evaluate generalization to unseen steganographic techniques, a Leave-One-Technique-Out (LOTO) evaluation protocol was employed. In each of the three splits, a fresh model instance was trained from scratch on two steganographic techniques alongside normal traffic, then evaluated against the third technique which was entirely withheld from training. The proposed FeatureTokenTransformer achieved a mean binary detection accuracy of **94.22%** and a mean protocol classification accuracy of **94.27%** across all three splits. These results demonstrate that the model captures generalizable representations of covert network behaviour rather than overfitting to the statistical patterns of specific techniques, satisfying the requirement for a technique-agnostic steganography detector.